# Head DLA Features

Global Direct Logit Attribution (DLA) analysis for Sudoku attention heads.

This notebook extracts every attention head output at a destination token, computes direct logit attribution to the final vocabulary, and summarizes influence over cells and digits.

In [ ]:
from pathlib import Path
import os, sys

# Allow running from repo root or from notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

## Setup

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from sudoku.activations import load_probe_dataset, load_checkpoint, derive_n_clues
from sudoku.data import PAD_TOKEN, SEP_TOKEN, decode_fill
from sudoku.data_bt import PUSH_TOKEN, POP_TOKEN

## Configuration And Load

In [ ]:
CACHE_PATH = "results/3M-backtracking-packing/activations.npz"
CKPT_DIR = "results/3M-backtracking-packing/checkpoint"

N_PUZZLES = 6400
BATCH_SIZE = 32
QUERY_STEP = 0  # 0 = SEP token, 1 = first trace token after SEP, ...

_, puzzles, sequences, n_clues, _ = load_probe_dataset(CACHE_PATH)
params, model = load_checkpoint(CKPT_DIR)

cfg = model.config
n_layers = cfg.n_layers
n_heads = cfg.n_heads
d_model = cfg.d_model
T = cfg.max_seq_len
head_dim = d_model // n_heads

print(f"Model: n_layers={n_layers}, n_heads={n_heads}, d_model={d_model}, T={T}")

if n_clues is None:
    n_clues = derive_n_clues(puzzles)

n_total = min(N_PUZZLES, len(sequences)) if N_PUZZLES else len(sequences)
query_label = "SEP" if QUERY_STEP == 0 else f"SEP+{QUERY_STEP}"
print(f"Using {n_total} puzzles; query={query_label}")

## Manual Forward Pass

In [ ]:
def _ln(x, scale, bias, eps=1e-5):
    m = x.mean(-1, keepdims=True)
    v = x.var(-1, keepdims=True)
    return (x - m) / jnp.sqrt(v + eps) * scale + bias


@jax.jit
def forward_gather_all_heads(params, tokens, query_pos):
    """Batched forward pass extracting per-head outputs at query_pos."""
    B = tokens.shape[0]

    tok_emb = params["token_emb"]["embedding"][tokens]
    pos_emb = params["pos_emb"]["embedding"][jnp.arange(T)[None, :]]
    x = tok_emb + pos_emb  # (B, T, d_model)

    head_outs_q_layers = []
    post_attn_q_layers = []
    b_idx = jnp.arange(B)

    for li in range(n_layers):
        blk = params[f"block_{li}"]
        attn = blk["CausalSelfAttention_0"]

        h = _ln(x, blk["LayerNorm_0"]["scale"], blk["LayerNorm_0"]["bias"])

        qkv = h @ attn["qkv"]["kernel"] + attn["qkv"]["bias"]
        q = qkv[..., :d_model].reshape(B, T, n_heads, head_dim).transpose(0, 2, 1, 3)
        k = qkv[..., d_model:2 * d_model].reshape(B, T, n_heads, head_dim).transpose(0, 2, 1, 3)
        v = qkv[..., 2 * d_model:].reshape(B, T, n_heads, head_dim).transpose(0, 2, 1, 3)

        scores = (q @ k.transpose(0, 1, 3, 2)) * (head_dim ** -0.5)
        mask = jnp.tril(jnp.ones((T, T), dtype=bool))
        scores = jnp.where(mask[None, None], scores, jnp.finfo(scores.dtype).min)
        w = jax.nn.softmax(scores, axis=-1)
        z = w @ v  # (B, n_heads, T, head_dim)

        W_O = attn["proj"]["kernel"]
        W_O_h = W_O.reshape(n_heads, head_dim, d_model)

        z_q = z[b_idx[:, None], jnp.arange(n_heads)[None, :], query_pos[:, None]]
        head_out_q = jnp.einsum("bhd,hde->bhe", z_q, W_O_h)
        head_outs_q_layers.append(head_out_q)

        z_cat = z.transpose(0, 2, 1, 3).reshape(B, T, d_model)
        x = x + z_cat @ W_O + attn["proj"]["bias"]
        post_attn_q_layers.append(x[b_idx, query_pos])

        h2 = _ln(x, blk["LayerNorm_1"]["scale"], blk["LayerNorm_1"]["bias"])
        h2 = jax.nn.gelu(h2 @ blk["Dense_0"]["kernel"] + blk["Dense_0"]["bias"])
        h2 = h2 @ blk["Dense_1"]["kernel"] + blk["Dense_1"]["bias"]
        x = x + h2

    head_outs_q = jnp.stack(head_outs_q_layers, axis=1)  # (B, n_layers, n_heads, d_model)
    post_attn_q = jnp.stack(post_attn_q_layers, axis=1)  # (B, n_layers, d_model)
    resid_final = x[b_idx, query_pos]                    # (B, d_model)
    return head_outs_q, post_attn_q, resid_final

## Collect Head Outputs

In [ ]:
head_outs_all = np.zeros((n_total, n_layers, n_heads, d_model), dtype=np.float32)
post_attn_all = np.zeros((n_total, n_layers, d_model), dtype=np.float32)
resid_final_all = np.zeros((n_total, d_model), dtype=np.float32)
valid_mask = np.zeros(n_total, dtype=bool)
query_pos_all = np.zeros(n_total, dtype=np.int32)

for batch_start in tqdm(range(0, n_total, BATCH_SIZE), desc=f"Forward Pass ({query_label})"):
    batch_end = min(batch_start + BATCH_SIZE, n_total)
    real_B = batch_end - batch_start

    tokens_np = np.full((BATCH_SIZE, T), PAD_TOKEN, dtype=np.int32)
    qpos_np = np.zeros(BATCH_SIZE, dtype=np.int32)
    local_valid = np.zeros(BATCH_SIZE, dtype=bool)

    for i in range(real_B):
        seq = sequences[batch_start + i]
        L = min(len(seq), T)
        tokens_np[i, :L] = seq[:L]
        nc = int(n_clues[batch_start + i])
        qp = nc + QUERY_STEP

        if nc < T and tokens_np[i, nc] == SEP_TOKEN and qp < T and tokens_np[i, qp] != PAD_TOKEN:
            qpos_np[i] = qp
            local_valid[i] = True

    h_q, p_q, r_f = forward_gather_all_heads(params, jnp.array(tokens_np), jnp.array(qpos_np))

    head_outs_all[batch_start:batch_end] = np.asarray(h_q)[:real_B]
    post_attn_all[batch_start:batch_end] = np.asarray(p_q)[:real_B]
    resid_final_all[batch_start:batch_end] = np.asarray(r_f)[:real_B]
    valid_mask[batch_start:batch_end] = local_valid[:real_B]
    query_pos_all[batch_start:batch_end] = qpos_np[:real_B]

n_valid = int(valid_mask.sum())
print(f"Used {n_valid}/{n_total} puzzles (valid destination token).")

head_outs = head_outs_all[valid_mask]      # (N, L, H, d_model)
post_attn = post_attn_all[valid_mask]      # (N, L, d_model)
resid_final = resid_final_all[valid_mask]  # (N, d_model)
qpos_valid = query_pos_all[valid_mask]
valid_seq_idx = np.where(valid_mask)[0]

## Compute DLA

In [ ]:
print("Computing Direct Logit Attribution (DLA)...")

W_U = np.asarray(params["lm_head"]["kernel"], dtype=np.float32)       # (d_model, vocab)
ln_scale = np.asarray(params["LayerNorm_0"]["scale"], dtype=np.float32)
ln_W = ln_scale[:, None] * W_U

resid_std = resid_final.std(axis=-1, keepdims=True)[:, None, None, :]  # (N, 1, 1, 1)
head_outs_centered = head_outs - head_outs.mean(axis=-1, keepdims=True)

dla = (head_outs_centered / resid_std) @ ln_W  # (N, L, H, vocab)
dla_placements = dla[..., :729]
dla_grid = dla_placements.reshape(n_valid, n_layers, n_heads, 9, 9, 9)

dla_influence = np.mean(np.abs(dla_grid), axis=0)  # (L, H, row, col, digit)

influence_cells = np.mean(dla_influence, axis=-1)       # (L, H, 9, 9)
influence_cells_2d = influence_cells.reshape(n_layers * n_heads, 81)

influence_digits = np.mean(dla_influence, axis=(2, 3))  # (L, H, 9)
influence_digits_2d = influence_digits.reshape(n_layers * n_heads, 9)

print("DLA shapes:")
print("  influence_cells:", influence_cells.shape)
print("  influence_digits:", influence_digits.shape)

## Plot Helpers

In [ ]:
def plot_head_grid(data, title, outfile=None, is_cells=True):
    """Plot an (n_layers x n_heads) grid of per-head min-max normalized heatmaps."""
    fig, axes = plt.subplots(
        n_layers,
        n_heads,
        figsize=(n_heads * 1.5, n_layers * 1.5),
        squeeze=False,
    )
    cmap = "Reds"

    for li in range(n_layers):
        for hi in range(n_heads):
            ax = axes[li, hi]
            g = data[li, hi]

            g_min = np.min(g)
            g_max = np.max(g)
            if g_max > g_min:
                g_norm = (g - g_min) / (g_max - g_min)
            else:
                g_norm = np.zeros_like(g)

            ax.imshow(g_norm, vmin=0, vmax=1.0, cmap=cmap, interpolation="nearest")
            ax.set_xticks([])
            ax.set_yticks([])

            if is_cells:
                for v in (2.5, 5.5):
                    ax.axvline(v, color="black", linewidth=0.8)
                    ax.axhline(v, color="black", linewidth=0.8)

            if li == 0:
                ax.set_title(f"H{hi}", fontsize=10)
            if hi == 0:
                ax.set_ylabel(f"L{li}", fontsize=10)

    fig.suptitle(title + "\n(Individually Min-Max Normalized per Head)", fontsize=14, y=1.02)
    fig.tight_layout()
    if outfile:
        fig.savefig(outfile, dpi=150, bbox_inches="tight")
        print(f"Saved {outfile}")
    plt.show()

## Generate Plots

In [ ]:
plot_head_grid(
    influence_cells,
    f"Targeting Influence (Mean Absolute DLA per Cell) at [{query_label}]",
    "head_dla_cells_grid.png",
    is_cells=True,
)

influence_digits_3x3 = influence_digits.reshape(n_layers, n_heads, 3, 3)
plot_head_grid(
    influence_digits_3x3,
    f"Digit Routing Influence (Mean Absolute DLA per Digit 1-9) at [{query_label}]\n"
    "Read left-to-right, top-to-bottom: 1,2,3 / 4,5,6 / 7,8,9",
    "head_dla_digits_grid.png",
    is_cells=False,
)

## OV Follow-Up

Run the more detailed `head_ov_experiment.py` routine for a selected subset of heads. Edit `HEADS_TO_INSPECT` after looking at the global DLA plots above.

In [ ]:
# Edit this list to inspect heads surfaced by the global DLA scan.
# Keeping this small avoids generating a large number of plots/text files.
HEADS_TO_INSPECT = [(0, 0)]

TOP_K = 15
PROBE_MIN_POS = 20
OUTPUT_DIR = Path("playground/head_ov_from_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Heads to inspect:", HEADS_TO_INSPECT)
print("Writing OV follow-up artifacts to", OUTPUT_DIR)

### Board-State Targets

In [ ]:
def grid_at_pos(seq, pos):
    """Board after consuming seq[:pos+1], respecting push/pop. Returns 81 digits; 0 means empty."""
    board = [0] * 81
    stack: list[list[int]] = []
    for p in range(min(pos + 1, len(seq))):
        tok = int(seq[p])
        if 0 <= tok <= 728:
            r, c, d = decode_fill(tok)
            board[r * 9 + c] = d
        elif tok == PUSH_TOKEN:
            stack.append(board[:])
        elif tok == POP_TOKEN and stack:
            board = stack.pop()
    return board


def box_digit_matrix(board):
    m = np.zeros((9, 9), dtype=np.int8)
    for cell, digit in enumerate(board):
        if digit == 0:
            continue
        r, c = divmod(cell, 9)
        box = (r // 3) * 3 + c // 3
        m[box, digit - 1] = 1
    return m


def row_digit_matrix(board):
    m = np.zeros((9, 9), dtype=np.int8)
    for cell, digit in enumerate(board):
        if digit:
            m[cell // 9, digit - 1] = 1
    return m


def col_digit_matrix(board):
    m = np.zeros((9, 9), dtype=np.int8)
    for cell, digit in enumerate(board):
        if digit:
            m[cell % 9, digit - 1] = 1
    return m


print("Building box/row/col-digit targets...")
Y_box = np.zeros((n_valid, 9, 9), dtype=np.int8)
Y_row = np.zeros((n_valid, 9, 9), dtype=np.int8)
Y_col = np.zeros((n_valid, 9, 9), dtype=np.int8)
for k, (seq_i, qp) in enumerate(zip(valid_seq_idx, qpos_valid)):
    board = grid_at_pos(sequences[int(seq_i)], int(qp))
    Y_box[k] = box_digit_matrix(board)
    Y_row[k] = row_digit_matrix(board)
    Y_col[k] = col_digit_matrix(board)

r_grid_2d = np.arange(9)[:, None].repeat(9, axis=1)
c_grid_2d = np.arange(9)[None, :].repeat(9, axis=0)
b_grid_2d = (r_grid_2d // 3) * 3 + (c_grid_2d // 3)

print("Targets:", Y_box.shape, Y_row.shape, Y_col.shape)

### OV Helpers

In [ ]:
resid_std_for_ov = resid_final.std(axis=-1, keepdims=True)
ln_bias = np.asarray(params["LayerNorm_0"]["bias"], dtype=np.float32)


def dla_of(component):
    """component: (N, d_model) -> (N, vocab). Final-LN stats are frozen to the full residual."""
    c = component - component.mean(axis=-1, keepdims=True)
    return (c / resid_std_for_ov) @ ln_W


def plot_dla_digits(mean_per_fill, outfile, title):
    """mean_per_fill: (9 rows, 9 cols, 9 digits). One 9x9 subplot per digit."""
    vmax = float(np.abs(mean_per_fill).max())
    if vmax == 0:
        vmax = 1.0
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for d in range(9):
        ax = axes[d // 3, d % 3]
        im = ax.imshow(mean_per_fill[:, :, d], cmap="RdBu_r", vmin=-vmax, vmax=vmax, interpolation="nearest")
        ax.set_title(f"digit {d + 1}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        for v in (2.5, 5.5):
            ax.axvline(v, color="k", lw=0.5)
            ax.axhline(v, color="k", lw=0.5)
    fig.suptitle(title, fontsize=10)
    fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, label="mean DLA")
    fig.savefig(outfile, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved {outfile}")


def top_bottom_tokens(mean_logit, k):
    fill = mean_logit[:729]
    top = np.argsort(fill)[-k:][::-1]
    bot = np.argsort(fill)[:k]
    lines = ["Top-k promoted fill tokens:"]
    for tok in top:
        r, c, d = decode_fill(int(tok))
        lines.append(f"  R{r + 1}C{c + 1}={d}   {fill[tok]:+.4f}")
    lines.append("Bottom-k suppressed fill tokens:")
    for tok in bot:
        r, c, d = decode_fill(int(tok))
        lines.append(f"  R{r + 1}C{c + 1}={d}   {fill[tok]:+.4f}")
    if len(mean_logit) > 730:
        lines.append(f"SEP={mean_logit[729]:+.4f}  PAD={mean_logit[730]:+.4f}")
    if len(mean_logit) > 733:
        lines.append(f"PUSH={mean_logit[731]:+.4f}  POP={mean_logit[732]:+.4f}  SUCCESS={mean_logit[733]:+.4f}")
    return "\n".join(lines)


def fit_probe(X, Y, test_idx, train_idx):
    """Fit one logistic regression per (structure, digit). Returns a (9, 9) AUC matrix."""
    aucs = np.full((9, 9), np.nan)
    for s in range(9):
        for d in range(9):
            y = Y[:, s, d]
            y_tr, y_te = y[train_idx], y[test_idx]
            if y_tr.sum() < PROBE_MIN_POS or y_tr.sum() > len(y_tr) - PROBE_MIN_POS:
                continue
            if y_te.sum() < 5 or y_te.sum() > len(y_te) - 5:
                continue
            clf = LogisticRegression(max_iter=1000, C=1.0).fit(X[train_idx], y_tr)
            pr = clf.predict_proba(X[test_idx])[:, 1]
            aucs[s, d] = roc_auc_score(y_te, pr)
    return aucs


def plot_probe_aucs(auc_head, auc_baseline, outfile, title, ylabel_prefix="box"):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
    for ax, data, name in ((axes[0], auc_head, "head alone"), (axes[1], auc_baseline, "post-attn baseline")):
        im = ax.imshow(data, vmin=0.5, vmax=1.0, cmap="viridis", interpolation="nearest")
        ax.set_title(name, fontsize=9)
        ax.set_xlabel("digit")
        ax.set_xticks(range(9))
        ax.set_xticklabels(range(1, 10))
        ax.set_yticks(range(9))
        ax.set_yticklabels([f"{ylabel_prefix} {i}" for i in range(9)])
        for s in range(9):
            for d in range(9):
                if not np.isnan(data[s, d]):
                    ax.text(d, s, f"{data[s, d]:.2f}", ha="center", va="center", fontsize=6, color="w" if data[s, d] < 0.75 else "k")
    fig.suptitle(title, fontsize=10)
    fig.colorbar(im, ax=axes, fraction=0.035, pad=0.02, label="AUC")
    fig.savefig(outfile, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved {outfile}")


def run_conditional_dla(dla_fill, mask_present, mean_logit, layer, head, axis_key, axis_label):
    dla_present = np.where(mask_present == 1, dla_fill, np.nan)
    dla_absent = np.where(mask_present == 0, dla_fill, np.nan)
    mean_present = np.nanmean(dla_present, axis=0)
    mean_absent = np.nanmean(dla_absent, axis=0)

    infix = "" if axis_key == "" else f"_{axis_key}"
    prefix = OUTPUT_DIR / f"head_ov_L{layer}H{head}"
    plot_dla_digits(mean_present, f"{prefix}_dla_digits{infix}_present.png", f"DLA digit PRESENT in {axis_label} L{layer}H{head} at {query_label}")
    plot_dla_digits(mean_absent, f"{prefix}_dla_digits{infix}_absent.png", f"DLA digit ABSENT in {axis_label} L{layer}H{head} at {query_label}")

    def full_vocab(cond_fill):
        out = mean_logit.copy()
        out[:729] = cond_fill.reshape(729)
        return out

    present_txt = Path(f"{prefix}_dla_top{infix}_present.txt")
    absent_txt = Path(f"{prefix}_dla_top{infix}_absent.txt")
    present_txt.write_text(top_bottom_tokens(full_vocab(mean_present), TOP_K) + "\n")
    absent_txt.write_text(top_bottom_tokens(full_vocab(mean_absent), TOP_K) + "\n")
    print(f"saved {present_txt}")
    print(f"saved {absent_txt}")

### Run OV Follow-Up

In [ ]:
rng = np.random.default_rng(42)
perm = rng.permutation(n_valid)
n_test = max(1, n_valid // 5)
test_idx = perm[:n_test]
train_idx = perm[n_test:]

_baseline_cache: dict[tuple[int, str], np.ndarray] = {}


def baseline_probe(layer, axis_key, Y):
    key = (layer, axis_key)
    if key not in _baseline_cache:
        _baseline_cache[key] = fit_probe(post_attn[:, layer, :], Y, test_idx, train_idx)
    return _baseline_cache[key]


AXES = [
    ("", "BOX", Y_box, b_grid_2d),
    ("row", "ROW", Y_row, r_grid_2d),
    ("col", "COL", Y_col, c_grid_2d),
]

for layer, head in HEADS_TO_INSPECT:
    print(f"\n=== Layer {layer} Head {head} ===")
    head_out = head_outs[:, layer, head, :]

    dla_head = dla_of(head_out)
    mean_logit = dla_head.mean(axis=0)
    mean_fill = mean_logit[:729].reshape(9, 9, 9)

    prefix = OUTPUT_DIR / f"head_ov_L{layer}H{head}"
    plot_dla_digits(mean_fill, f"{prefix}_dla_digits.png", f"DLA of L{layer}H{head} at {query_label} ({n_valid} puzzles)")

    top_txt = Path(f"{prefix}_dla_top.txt")
    top_txt.write_text(top_bottom_tokens(mean_logit, TOP_K) + "\n")
    print(f"saved {top_txt}")

    dla_fill = dla_head[:, :729].reshape(-1, 9, 9, 9)

    for axis_key, axis_label, Y_struct, idx_grid in AXES:
        auc_head = fit_probe(head_out, Y_struct, test_idx, train_idx)
        auc_baseline = baseline_probe(layer, axis_key, Y_struct)

        probe_file = f"{prefix}_probe.png" if axis_key == "" else f"{prefix}_probe_{axis_key}.png"
        plot_probe_aucs(
            auc_head,
            auc_baseline,
            probe_file,
            f"{axis_label}-digit probe AUC - L{layer}H{head} (head alone vs layer-{layer} post-attn)",
            ylabel_prefix=axis_label.lower(),
        )
        head_mean = np.nanmean(auc_head)
        base_mean = np.nanmean(auc_baseline)
        print(f"[{axis_label}] head-alone mean AUC: {head_mean:.3f}; baseline: {base_mean:.3f}; delta {head_mean - base_mean:+.3f}")

        mask_present = Y_struct[:, idx_grid, :]
        run_conditional_dla(dla_fill, mask_present, mean_logit, layer, head, axis_key, axis_label)

print("Done.")